In [4]:
!pip install -q "lerobot[smolvla,dataset]==0.7.0"

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, torch

PROJ = "/content/drive/MyDrive/257_project"
MODEL_DIR = f"{PROJ}/smolvla_libero"
CKPT_DIR = f"{PROJ}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
os.makedirs(f"{PROJ}/results", exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
print("model on Drive:", os.path.exists(MODEL_DIR))

Mounted at /content/drive
GPU: NVIDIA L4
model on Drive: True


In [10]:
import random, time as timer, torch, pandas as pd
import av
from lerobot.utils import import_utils
import_utils._require_package_cache["av"] = True
from lerobot.datasets import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.policies.factory import make_pre_post_processors
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# settings
STEPS = 1000          # training steps
BATCH = 8
LR = 2e-5
LAM = 0.0             # weight of the vision-only (prior) branch
BETA = 1.0            # weight of the PMI term
MARGIN = 0.3          # target PMI
N_TRAIN_EPISODES = 60 # per suite
NEG = "swap"
NEG_GRAD = True
SMOKE = False

REPOS = {"goal": "lerobot/libero_goal_image", "object": "lerobot/libero_object_image"}
# Phase 1 evaluation episodes: never trained on
EVAL_EPISODES = {
    "goal": [20, 132, 197, 207, 215, 248, 261, 388, 401, 424],
    "object": [48, 71, 111, 144, 155, 183, 244, 258, 298, 386],
}
device = "cuda"
random.seed(1); torch.manual_seed(1)
print("ready")

ready


In [11]:
policy = SmolVLAPolicy.from_pretrained(MODEL_DIR).to(device)
pre, post = make_pre_post_processors(
    policy.config, pretrained_path=MODEL_DIR,
    preprocessor_overrides={"device_processor": {"device": device}},
)
CHUNK = policy.config.chunk_size

params = [p for p in policy.parameters() if p.requires_grad]
print(f"trainable: {sum(p.numel() for p in params)/1e6:.1f}M of {sum(p.numel() for p in policy.parameters())/1e6:.1f}M")
print("chunk size:", CHUNK)

Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Loading weights from local directory
trainable: 97.5M of 604.9M
chunk size: 50


In [12]:
class PolicyKeys(torch.utils.data.Dataset):
    """Rename the wrist camera to the name SmolVLA expects."""
    def __init__(self, ds): self.ds = ds
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        item = dict(self.ds[i])
        item["observation.images.image2"] = item.pop("observation.images.wrist_image")
        return item

n_eps = 2 if SMOKE else N_TRAIN_EPISODES
parts, SWAP = [], {}
for suite, repo in REPOS.items():
    root = f"/content/data/{repo.replace('/', '__')}"
    meta = LeRobotDatasetMetadata(repo, root=root)
    tasks = meta.tasks.sort_values("task_index").index.tolist()
    SWAP.update({t: tasks[(i + 1) % len(tasks)] for i, t in enumerate(tasks)})   # each task -> next task, same suite
    pool = [e for e in range(meta.total_episodes) if e not in EVAL_EPISODES[suite]]
    eps = sorted(random.sample(pool, n_eps))
    ds = LeRobotDataset(repo, root=root, episodes=eps,
                        delta_timestamps={"action": [i / meta.fps for i in range(CHUNK)]})
    got = set(int(e) for e in ds.hf_dataset["episode_index"])
    assert got == set(eps), f"{suite}: episode mismatch"
    assert not got & set(EVAL_EPISODES[suite]), f"{suite}: eval episode leaked into training"
    parts.append(PolicyKeys(ds))
    print(f"{suite}: {len(ds)} frames from {len(eps)} episodes")

loader = torch.utils.data.DataLoader(
    torch.utils.data.ConcatDataset(parts), batch_size=2 if SMOKE else BATCH,
    shuffle=True, num_workers=2, drop_last=True,
)
print("swap map size:", len(SWAP))
print("example:", list(SWAP.items())[0])
print("batches per pass:", len(loader))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 41 files:   0%|          | 0/41 [00:00<?, ?it/s]

goal: 7129 frames from 60 episodes


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 47 files:   0%|          | 0/47 [00:00<?, ?it/s]

object: 8839 frames from 60 episodes
swap map size: 20
example: ('put the bowl on the plate', 'put the wine bottle on the rack')
batches per pass: 1996


In [ ]:
steps = 3 if SMOKE else STEPS
log_every = 1 if SMOKE else 25
opt = torch.optim.AdamW(params, lr=LR, weight_decay=1e-10)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=steps)
neg_needs_grad = LAM > 0 or NEG_GRAD

policy.train()
log, step, t0 = [], 0, timer.time()
while step < steps:
    for batch in loader:
        if step >= steps:
            break
        tasks = list(batch["task"])
        B = len(tasks)
        noise = torch.randn(B, CHUNK, policy.config.max_action_dim, device=device)
        t = torch.rand(B, device=device) * 0.998 + 0.001

        with torch.autocast("cuda", dtype=torch.bfloat16):
            # branch 1: image + RIGHT instruction
            b_post = dict(batch); b_post["task"] = tasks
            loss_post, _ = policy.forward(pre(b_post), noise=noise, time=t, reduction="none")
            # branch 2: image + WRONG instruction (or blank)
            b_neg = dict(batch)
            b_neg["task"] = [SWAP[x] for x in tasks] if NEG == "swap" else [""] * B
            with torch.set_grad_enabled(neg_needs_grad):
                loss_neg, _ = policy.forward(pre(b_neg), noise=noise, time=t, reduction="none")

        loss_post, loss_neg = loss_post.float(), loss_neg.float()
        neg = loss_neg if NEG_GRAD else loss_neg.detach()
        gap = neg - loss_post                                  # want this >= MARGIN
        hinge = torch.clamp(MARGIN - gap, min=0).mean()
        loss = (1 - LAM) * loss_post.mean() + LAM * loss_neg.mean() + BETA * hinge

        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 10.0)
        opt.step(); sched.step(); step += 1

        log.append({"step": step, "loss": loss.item(), "loss_post": loss_post.mean().item(),
                    "loss_neg": loss_neg.mean().item(), "gap": gap.mean().item(), "hinge": hinge.item()})
        if step % log_every == 0 or step == 1:
            r = log[-1]
            print(f"step {step}/{steps} | loss {r['loss']:.3f} | post {r['loss_post']:.3f} "
                  f"| neg {r['loss_neg']:.3f} | gap {r['gap']:.3f} | hinge {r['hinge']:.3f} | {timer.time()-t0:.0f}s")
print("done")

step 1/1000 | loss 0.462 | post 0.341 | neg 0.747 | gap 0.405 | hinge 0.121 | 1s
step 25/1000 | loss 1.094 | post 0.698 | neg 0.961 | gap 0.263 | hinge 0.396 | 24s
step 50/1000 | loss 0.589 | post 0.491 | neg 1.289 | gap 0.798 | hinge 0.098 | 47s
step 75/1000 | loss 0.523 | post 0.395 | neg 0.614 | gap 0.219 | hinge 0.128 | 70s
step 100/1000 | loss 0.753 | post 0.602 | neg 0.894 | gap 0.292 | hinge 0.151 | 94s
step 125/1000 | loss 0.389 | post 0.320 | neg 0.811 | gap 0.491 | hinge 0.069 | 117s
step 150/1000 | loss 0.411 | post 0.330 | neg 1.383 | gap 1.053 | hinge 0.081 | 140s
step 175/1000 | loss 0.685 | post 0.515 | neg 0.732 | gap 0.217 | hinge 0.170 | 163s
step 200/1000 | loss 0.521 | post 0.383 | neg 0.618 | gap 0.235 | hinge 0.139 | 187s
step 225/1000 | loss 0.474 | post 0.344 | neg 0.653 | gap 0.310 | hinge 0.131 | 210s
step 250/1000 | loss 0.523 | post 0.361 | neg 0.656 | gap 0.295 | hinge 0.162 | 233s
step 275/1000 | loss 0.628 | post 0.520 | neg 0.964 | gap 0.445 | hinge 0.10

In [14]:
print(len(log), "steps in memory")

1000 steps in memory


In [15]:
import glob, shutil
RUN_NAME = "swap_hinge"     # change to "control" for the control run
out_dir = f"{CKPT_DIR}/{RUN_NAME}"
policy.save_pretrained(out_dir)
for f in glob.glob(f"{MODEL_DIR}/policy_*processor*"):
    shutil.copy(f, out_dir)       # the eval code needs the processors too
pd.DataFrame(log).to_csv(f"{out_dir}/train_log.csv", index=False)
print("saved to", out_dir)

saved to /content/drive/MyDrive/257_project/checkpoints/swap_hinge


In [16]:
import numpy as np, gc

EVAL_REPOS = {"goal": "lerobot/libero_goal_image", "object": "lerobot/libero_object_image",
              "spatial": "lerobot/libero_spatial_image", "long": "lerobot/libero_10_image"}

def evaluate(model_dir, name, n_eps=10, n_frames=200, batch=16):
    random.seed(0); torch.manual_seed(0)      # same seed as Phase 1, so the same episodes and frames
    pol = SmolVLAPolicy.from_pretrained(model_dir).to(device).eval()
    pre_e, _ = make_pre_post_processors(pol.config, pretrained_path=model_dir,
                                        preprocessor_overrides={"device_processor": {"device": device}})
    dsets, swap = {}, {}
    for suite, repo in EVAL_REPOS.items():
        root = f"/content/data/{repo.replace('/', '__')}"
        m = LeRobotDatasetMetadata(repo, root=root)
        tasks = m.tasks.sort_values("task_index").index.tolist()
        swap.update({t: tasks[(i + 1) % len(tasks)] for i, t in enumerate(tasks)})
        eps = sorted(random.sample(range(m.total_episodes), n_eps))
        dsets[suite] = PolicyKeys(LeRobotDataset(repo, root=root, episodes=eps,
                         delta_timestamps={"action": [i / m.fps for i in range(CHUNK)]}))
        print(f"{name} | {suite}: episodes {eps}")
    rows = []
    for suite, ds in dsets.items():
        idx = random.sample(range(len(ds)), n_frames)
        for b in range(0, len(idx), batch):
            part = idx[b:b + batch]
            bt = torch.utils.data.default_collate([ds[i] for i in part])
            true_t = list(bt["task"])
            conds = {"true": true_t, "blank": [""] * len(part), "swap": [swap[x] for x in true_t]}
            noise = torch.randn(len(part), CHUNK, pol.config.max_action_dim, device=device)
            tt = torch.rand(len(part), device=device) * 0.998 + 0.001
            out = {}
            with torch.no_grad():
                for c, tl in conds.items():
                    bb = dict(bt); bb["task"] = tl
                    loss, _ = pol.forward(pre_e(bb), noise=noise, time=tt, reduction="none")
                    out[c] = loss.float().cpu().tolist()
            for k, i in enumerate(part):
                rows.append({"model": name, "suite": suite, "frame": i, "task": true_t[k],
                             **{f"loss_{c}": out[c][k] for c in conds}})
        print(f"{name} | {suite}: done")
    df = pd.DataFrame(rows)
    df.to_csv(f"{PROJ}/results/eval_{name}.csv", index=False)
    del pol; gc.collect(); torch.cuda.empty_cache()
    return df

print("evaluate() ready")

evaluate() ready


In [17]:
try:
    del policy
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

try:
    all_df
except NameError:
    all_df = pd.read_csv(f"{PROJ}/results/eval_all.csv")

NEW = "swap_hinge"
all_df = all_df[all_df.model != NEW]
all_df = pd.concat([all_df, evaluate(f"{CKPT_DIR}/{NEW}", NEW)], ignore_index=True)
all_df.to_csv(f"{PROJ}/results/eval_all.csv", index=False)

all_df["gap_swap"] = all_df.loss_swap - all_df.loss_true
all_df["gap_blank"] = all_df.loss_blank - all_df.loss_true
rng = np.random.default_rng(0)
def ci(x, n=2000):
    m = [rng.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(m, [2.5, 97.5])

show = ["baseline", "control", NEW]
rows = []
for (model, suite), g in all_df[all_df.model.isin(show)].groupby(["model", "suite"]):
    s, b = ci(g.gap_swap.values), ci(g.gap_blank.values)
    rows.append({"suite": suite, "model": model, "o": show.index(model),
                 "loss_true": round(g.loss_true.mean(), 3),
                 "gap_swap": f"{g.gap_swap.mean():.3f} [{s[0]:.3f}, {s[1]:.3f}]",
                 "gap_blank": f"{g.gap_blank.mean():.3f} [{b[0]:.3f}, {b[1]:.3f}]"})
print(pd.DataFrame(rows).sort_values(["suite", "o"]).drop(columns="o").to_string(index=False))

print("\nChange vs baseline (same frames):")
base = all_df[all_df.model == "baseline"].set_index(["suite", "frame"])
for model in ["control", NEW]:
    m = all_df[all_df.model == model].set_index(["suite", "frame"])
    for suite in ["object", "goal", "spatial", "long"]:
        ds_ = (m.loc[suite].gap_swap - base.loc[suite].gap_swap).values
        db_ = (m.loc[suite].gap_blank - base.loc[suite].gap_blank).values
        dt_ = (m.loc[suite].loss_true - base.loc[suite].loss_true).values
        cs, cb = ci(ds_), ci(db_)
        print(f"  {model:12s} {suite:8s} | Δgap_swap {ds_.mean():+.3f} [{cs[0]:+.3f}, {cs[1]:+.3f}]"
              f" | Δgap_blank {db_.mean():+.3f} [{cb[0]:+.3f}, {cb[1]:+.3f}] | Δloss_true {dt_.mean():+.3f}")

Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Loading weights from local directory


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

swap_hinge | goal: episodes [20, 132, 197, 207, 215, 248, 261, 388, 401, 424]


... (9 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

swap_hinge | long: episodes [37, 51, 161, 169, 181, 222, 241, 286, 350, 373]
swap_hinge | goal: done
swap_hinge | object: done
swap_hinge | spatial: done
swap_hinge | long: done
  suite      model  loss_true             gap_swap            gap_blank
   goal   baseline      0.455 0.715 [0.614, 0.829] 0.521 [0.459, 0.582]
   goal    control      0.469 0.739 [0.636, 0.854] 0.532 [0.468, 0.598]
   goal swap_hinge      0.482 1.069 [0.953, 1.193] 0.659 [0.588, 0.735]
   long   baseline      0.370 0.243 [0.203, 0.288] 0.301 [0.267, 0.335]
   long    control      0.375 0.255 [0.212, 0.301] 0.297 [0.263, 0.332]
   long swap_hinge      0.389 0.337 [0.289, 0.389] 0.356 [0.315, 0.394]
 object   baseline      0.364 0.085 [0.066, 0.105] 0.307 [0.272, 0.347]
 object    control      0.371 0.080 [0.061, 0.099] 0.299 [0.261, 0.338]
 object swap_hinge      0.395 0.246 [0.211, 0.283] 0.371 [0.332, 0.410]
spatial   baseline      0.512 0.295 [0.268, 0.324] 0.425 [0.374, 0.478]
spatial    control      0.517 

In [ ]:
try:
    del policy                      # free the training model's GPU memory
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

models = {"baseline": MODEL_DIR,
          "control": f"{CKPT_DIR}/control",
          "langforce_lite": f"{CKPT_DIR}/langforce_lite"}
all_df = pd.concat([evaluate(path, name) for name, path in models.items()])
all_df.to_csv(f"{PROJ}/results/eval_all.csv", index=False)
print("all done")

Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Loading weights from local directory


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

baseline | goal: episodes [20, 132, 197, 207, 215, 248, 261, 388, 401, 424]


... (13 outputs left out)

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Loading weights from local directory
control | goal: episodes [20, 132, 197, 207, 215, 248, 261, 388, 401, 424]
control | object: episodes [48, 71, 111, 144, 155, 183, 244, 258, 298, 386]
control | spatial: episodes [50, 75, 128, 158, 272, 308, 316, 361, 409, 414]
control | long: episodes [37, 51, 161, 169, 181, 222, 241, 286, 350, 373]
control | goal: done
control | object: done
control | spatial: done
control | long: done
Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Loading weights from local directory
langforce_lite | goal: episodes [20, 132, 197, 207, 215, 248, 261, 388, 401, 424]
langforce_lite | object: episodes [48, 71, 111, 144, 155, 183, 244, 258, 298, 386]
langforce_lite | spatial: episodes [50, 75, 128, 158, 272, 308, 316, 361, 409, 414]
langforce_lite | long: episodes [37, 51, 161, 169, 181, 222, 241, 286, 350, 373]
langforce_lite | goal: done
langforce_lite | object: done
langforce_lite | spatial: done
langforce_lite | long: done
all done


In [ ]:
rng = np.random.default_rng(0)
def ci(x, n=2000):
    m = [rng.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(m, [2.5, 97.5])

all_df["gap_swap"] = all_df.loss_swap - all_df.loss_true
all_df["gap_blank"] = all_df.loss_blank - all_df.loss_true

# 1) main table: mean losses and gaps per model and suite
rows = []
for (model, suite), g in all_df.groupby(["model", "suite"]):
    s, b = ci(g.gap_swap.values), ci(g.gap_blank.values)
    rows.append({"suite": suite, "model": model,
                 "loss_true": round(g.loss_true.mean(), 3),
                 "gap_blank": f"{g.gap_blank.mean():.3f} [{b[0]:.3f}, {b[1]:.3f}]",
                 "gap_swap": f"{g.gap_swap.mean():.3f} [{s[0]:.3f}, {s[1]:.3f}]"})
order = {"baseline": 0, "control": 1, "langforce_lite": 2}
table = pd.DataFrame(rows).sort_values(["suite", "model"], key=lambda c: c.map(order) if c.name == "model" else c)
print(table.to_string(index=False))

# 2) paired change vs baseline (same frames), with 95% CI
print("\nChange vs baseline (same frames):")
base = all_df[all_df.model == "baseline"].set_index(["suite", "frame"])
for model in ["control", "langforce_lite"]:
    m = all_df[all_df.model == model].set_index(["suite", "frame"])
    for suite in ["object", "goal", "spatial", "long"]:
        d_blank = (m.loc[suite].gap_blank - base.loc[suite].gap_blank).values
        d_swap = (m.loc[suite].gap_swap - base.loc[suite].gap_swap).values
        d_true = (m.loc[suite].loss_true - base.loc[suite].loss_true).values
        cb, cs = ci(d_blank), ci(d_swap)
        print(f"  {model:15s} {suite:8s} | Δgap_blank {d_blank.mean():+.3f} [{cb[0]:+.3f}, {cb[1]:+.3f}]"
              f" | Δgap_swap {d_swap.mean():+.3f} [{cs[0]:+.3f}, {cs[1]:+.3f}] | Δloss_true {d_true.mean():+.3f}")

  suite          model  loss_true            gap_blank             gap_swap
   goal       baseline      0.455 0.521 [0.459, 0.582] 0.715 [0.614, 0.829]
   goal        control      0.469 0.532 [0.468, 0.598] 0.739 [0.636, 0.854]
   goal langforce_lite      0.472 0.242 [0.208, 0.277] 0.624 [0.528, 0.723]
   long       baseline      0.370 0.301 [0.267, 0.335] 0.243 [0.203, 0.288]
   long        control      0.375 0.297 [0.263, 0.332] 0.255 [0.212, 0.301]
   long langforce_lite      0.377 0.173 [0.148, 0.201] 0.235 [0.196, 0.280]
 object       baseline      0.364 0.307 [0.272, 0.347] 0.085 [0.066, 0.105]
 object        control      0.371 0.299 [0.261, 0.338] 0.080 [0.061, 0.099]
 object langforce_lite      0.370 0.068 [0.058, 0.078] 0.074 [0.056, 0.093]
spatial       baseline      0.512 0.425 [0.374, 0.478] 0.295 [0.268, 0.324]
spatial        control      0.517 0.431 [0.379, 0.488] 0.307 [0.277, 0.337]
spatial langforce_lite      0.519 0.287 [0.251, 0.326] 0.286 [0.258, 0.315]

Change vs b

In [ ]:
# --- evaluate pmi_only and add it to the comparison ---
try:
    del policy                      # free the training model's GPU memory
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

# reload earlier results if the session restarted
try:
    all_df
except NameError:
    all_df = pd.read_csv(f"{PROJ}/results/eval_all.csv")

all_df = all_df[all_df.model != "pmi_only"]                 # avoid duplicates if re-run
new_df = evaluate(f"{CKPT_DIR}/pmi_only", "pmi_only")
all_df = pd.concat([all_df, new_df], ignore_index=True)
all_df.to_csv(f"{PROJ}/results/eval_all.csv", index=False)

all_df["gap_swap"] = all_df.loss_swap - all_df.loss_true
all_df["gap_blank"] = all_df.loss_blank - all_df.loss_true
rng = np.random.default_rng(0)
def ci(x, n=2000):
    m = [rng.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return np.percentile(m, [2.5, 97.5])

# 1) main table
order = {"baseline": 0, "control": 1, "langforce_lite": 2, "pmi_only": 3}
rows = []
for (model, suite), g in all_df.groupby(["model", "suite"]):
    s, b = ci(g.gap_swap.values), ci(g.gap_blank.values)
    rows.append({"suite": suite, "model": model, "o": order[model],
                 "loss_true": round(g.loss_true.mean(), 3),
                 "gap_blank": f"{g.gap_blank.mean():.3f} [{b[0]:.3f}, {b[1]:.3f}]",
                 "gap_swap": f"{g.gap_swap.mean():.3f} [{s[0]:.3f}, {s[1]:.3f}]"})
table = pd.DataFrame(rows).sort_values(["suite", "o"]).drop(columns="o")
print(table.to_string(index=False))

# 2) paired change vs baseline (same frames)
print("\nChange vs baseline (same frames):")
base = all_df[all_df.model == "baseline"].set_index(["suite", "frame"])
for model in ["control", "langforce_lite", "pmi_only"]:
    m = all_df[all_df.model == model].set_index(["suite", "frame"])
    for suite in ["object", "goal", "spatial", "long"]:
        d_blank = (m.loc[suite].gap_blank - base.loc[suite].gap_blank).values
        d_swap = (m.loc[suite].gap_swap - base.loc[suite].gap_swap).values
        d_true = (m.loc[suite].loss_true - base.loc[suite].loss_true).values
        cb, cs = ci(d_blank), ci(d_swap)
        print(f"  {model:15s} {suite:8s} | Δgap_blank {d_blank.mean():+.3f} [{cb[0]:+.3f}, {cb[1]:+.3f}]"
              f" | Δgap_swap {d_swap.mean():+.3f} [{cs[0]:+.3f}, {cs[1]:+.3f}] | Δloss_true {d_true.mean():+.3f}")

Loading  HuggingFaceTB/SmolVLM2-500M-Instruct weights ...


Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Loading weights from local directory
pmi_only | goal: episodes [20, 132, 197, 207, 215, 248, 261, 388, 401, 424]
pmi_only | object: episodes [48, 71, 111, 144, 155, 183, 244, 258, 298, 386]
pmi_only | spatial: episodes [50, 75, 128, 158, 272, 308, 316, 361, 409, 414]
pmi_only | long: episodes [37, 51, 161, 169, 181, 222, 241, 286, 350, 373]
pmi_only | goal: done
pmi_only | object: done
pmi_only | spatial: done
pmi_only | long: done
  suite          model  loss_true            gap_blank             gap_swap
   goal       baseline      0.455 0.521 [0.459, 0.582] 0.715 [0.614, 0.829]
   goal        control      0.469 0.532 [0.468, 0.598] 0.739 [0.636, 0.854]
   goal langforce_lite      0.472 0.242 [0.208, 0.277] 0.624 [0.528, 0.723]
   goal       pmi_only      0.470 0.532 [0.468, 0.597] 0.732 [0.627, 0.848]
   long       baseline      0.370 0.301 [0.267, 0.335] 0.243 [0.203, 0.288]
   long        control      0.375 0.297 [0.263, 0.332] 0.255 [0.212, 0.301]
   long langforce_lite      0.37